In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing import image_dataset_from_directory
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing import image
from tensorflow.keras.layers import Input, RandomFlip, RandomRotation, RandomZoom, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
import numpy as np
import os

In [2]:
dataset_path = 'C:/Users/muiga/Downloads/archive/images'

In [3]:
img_size = (224, 224)
batch_size = 32

In [4]:
train_ds = image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=img_size,
    batch_size=batch_size
)

Found 20580 files belonging to 120 classes.
Using 16464 files for training.


In [5]:
val_ds = image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=img_size,
    batch_size=batch_size
)

Found 20580 files belonging to 120 classes.
Using 4116 files for validation.


In [6]:
class_names = train_ds.class_names
print(class_names)

['Afghan_hound', 'African_hunting_dog', 'Airedale', 'American_Staffordshire_terrier', 'Appenzeller', 'Australian_terrier', 'Bedlington_terrier', 'Bernese_mountain_dog', 'Blenheim_spaniel', 'Border_collie', 'Border_terrier', 'Boston_bull', 'Bouvier_des_Flandres', 'Brabancon_griffon', 'Brittany_spaniel', 'Cardigan', 'Chesapeake_Bay_retriever', 'Chihuahua', 'Dandie_Dinmont', 'Doberman', 'English_foxhound', 'English_setter', 'English_springer', 'EntleBucher', 'Eskimo_dog', 'French_bulldog', 'German_shepherd', 'German_short-haired_pointer', 'Gordon_setter', 'Great_Dane', 'Great_Pyrenees', 'Greater_Swiss_Mountain_dog', 'Ibizan_hound', 'Irish_setter', 'Irish_terrier', 'Irish_water_spaniel', 'Irish_wolfhound', 'Italian_greyhound', 'Japanese_spaniel', 'Kerry_blue_terrier', 'Labrador_retriever', 'Lakeland_terrier', 'Leonberg', 'Lhasa', 'Maltese_dog', 'Mexican_hairless', 'Newfoundland', 'Norfolk_terrier', 'Norwegian_elkhound', 'Norwich_terrier', 'Old_English_sheepdog', 'Pekinese', 'Pembroke', 'Po

In [10]:
# 1. Load the pre-trained MobileNetV2 base model
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

# 2. Build the model architecture
x = base_model.output
x = GlobalAveragePooling2D()(x)
predictions = Dense(120, activation='softmax')(x)  # 120 classes for dog breeds
model = Model(inputs=base_model.input, outputs=predictions)

# 3. Compile the model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 4. Fit the model
history = model.fit(train_ds, validation_data=val_ds, epochs=150)
print("\nTraining Complete!\n")

Epoch 1/150
515/515 ━━━━━━━━━━━━━━━━━━━━ 344s 658ms/step - accuracy: 0.0296 - loss: 4.8014 - val_accuracy: 0.0401 - val_loss: 4.6181
Epoch 2/150
515/515 ━━━━━━━━━━━━━━━━━━━━ 334s 648ms/step - accuracy: 0.0736 - loss: 4.3460 - val_accuracy: 0.0539 - val_loss: 4.5052
Epoch 3/150
515/515 ━━━━━━━━━━━━━━━━━━━━ 336s 652ms/step - accuracy: 0.1082 - loss: 4.0995 - val_accuracy: 0.0634 - val_loss: 4.4561
Epoch 4/150
515/515 ━━━━━━━━━━━━━━━━━━━━ 333s 646ms/step - accuracy: 0.1399 - loss: 3.9086 - val_accuracy: 0.0680 - val_loss: 4.4603
Epoch 5/150
515/515 ━━━━━━━━━━━━━━━━━━━━ 333s 647ms/step - accuracy: 0.1673 - loss: 3.7520 - val_accuracy: 0.0680 - val_loss: 4.4622
Epoch 6/150
515/515 ━━━━━━━━━━━━━━━━━━━━ 332s 644ms/step - accuracy: 0.1916 - loss: 3.6252 - val_accuracy: 0.0683 - val_loss: 4.4714
Epoch 7/150
515/515 ━━━━━━━━━━━━━━━━━━━━ 333s 647ms/step - accuracy: 0.2120 - loss: 3.5026 - val_accuracy: 0.0797 - val_loss: 4.4322
Epoch 8/150
515/515 ━━━━━━━━━━━━━━━━━━━━ 333s 646ms/step - accuracy: 

In [11]:
def predict_new_image(img_path):
    """Takes an image path, formats it, and returns the model's prediction."""
    if not os.path.exists(img_path):
        print("Error: Could not find that file. Check the path and try again.")
        return

    try:
        # Load and format the image to match what the model expects (224x224)
        img = image.load_img(img_path, target_size=img_size)
        img_array = image.img_to_array(img)
        img_array = tf.expand_dims(img_array, 0) # Create a batch of 1

        # Run the image through the model
        preds = model.predict(img_array, verbose=0)

        # Calculate results
        predicted_class_idx = np.argmax(preds[0])
        predicted_breed = class_names[predicted_class_idx]
        confidence = np.max(preds[0]) * 100

        print("\n" + "="*30)
        print(f"Predicted Breed: {predicted_breed}")
        print(f"Confidence:      {confidence:.2f}%")
        print("="*30 + "\n")

    except Exception as e:
        print(f"Error processing image: {e}")

# Interactive loop so you can test multiple images without retraining
while True:
    print("Type 'quit' to exit.")
    user_input = input("Enter the file path of a new dog picture to test: ")

    if user_input.lower() == 'quit':
        print("Exiting...")
        break

    predict_new_image(user_input)

Type 'quit' to exit.


Enter the file path of a new dog picture to test:  C:/Users/muiga/Downloads/dog1.jpg



Predicted Breed: Samoyed
Confidence:      70.92%

Type 'quit' to exit.


Enter the file path of a new dog picture to test:  C:/Users/muiga/Downloads/dog1.jpg



Predicted Breed: Tibetan_terrier
Confidence:      44.61%

Type 'quit' to exit.


...


Predicted Breed: miniature_pinscher
Confidence:      81.70%

Type 'quit' to exit.


Enter the file path of a new dog picture to test:  C:/Users/muiga/Downloads/dog1.jpeg



Predicted Breed: Scottish_deerhound
Confidence:      34.02%

Type 'quit' to exit.


Enter the file path of a new dog picture to test:  quit


Exiting...
